In [4]:
import numpy as np
from scipy.spatial import cKDTree
from umap.umap_ import fuzzy_simplicial_set, nearest_neighbors
from scipy.sparse import csr_matrix
from sklearn.preprocessing import StandardScaler
from scipy.special import expit
from adbench.myutils_new import Utils
from adbench.run_new import RunPipeline
import pandas as pd
from numba import njit, prange
from pynndescent import NNDescent


# -------------------------------------------------------
# Utility functions
# -------------------------------------------------------

def count_points_within_radius(X, tree, epsilon):
    neighbors = tree.query_ball_tree(tree, epsilon)
    counts = np.array([len(pts) - 1 for pts in neighbors])
    return counts


def max_min_distances_kdtree(X):
    tree = cKDTree(X)
    dists, _ = tree.query(X, k=X.shape[0])
    all_distances = dists[:, 1:].flatten()
    return np.max(all_distances), np.min(all_distances)


def binary_search_condition(low, high, condition, tol=1e-4, max_iter=50):
    result = None
    for _ in range(max_iter):
        mid = (low + high) / 2
        if condition(mid):
            result = mid
            high = mid
        else:
            low = mid
        if abs(high - low) < tol:
            break
    return result


def condition_formulation(point_in_radius_counts,
                           nbd_sample_count_threshold,
                           satisfiability_proportion):
    satisfied = (point_in_radius_counts > nbd_sample_count_threshold).astype(np.int32)
    return satisfied.sum() >= satisfiability_proportion



def get_empirical_weights(
    X,
    nbd_sample_count_threshold=5,
    max_iters_weight_count=4,   # kept for compatibility
    satisfiability_proportion=0.3,
    n_neighbors=15,
    metric='euclidean',
    random_state=42,
    batch_size=1000
):

    def umap_graph_similarity(X_batch):
        knn_indices, knn_dists, _ = nearest_neighbors(
            X_batch,
            n_neighbors=n_neighbors,
            metric=metric,
            metric_kwds={},
            angular=False,
            random_state=random_state,
            low_memory=True,
            use_pynndescent=True
        )
        G, _, _ = fuzzy_simplicial_set(
            X_batch,
            n_neighbors=n_neighbors,
            random_state=random_state,
            metric=metric,
            knn_indices=knn_indices,
            knn_dists=knn_dists,
            angular=False,
            set_op_mix_ratio=1.0,
            local_connectivity=1.0,
            apply_set_operations=True,
            verbose=False
        )
        return G.toarray() if isinstance(G, csr_matrix) else G

    # ---------------- SINGLE-RADIUS VERSION ----------------
    def compute_weights_from_similarity(sim, X_batch):
        tree = cKDTree(sim)

        max_dist, min_dist = max_min_distances_kdtree(sim)

        threshold = (
            max(1, len(X_batch) - 1)
            if nbd_sample_count_threshold >= len(X_batch)
            else nbd_sample_count_threshold
        )
        effective_required = int(satisfiability_proportion * len(X_batch))

        eps = binary_search_condition(
            min_dist, max_dist,
            lambda mid: condition_formulation(
                count_points_within_radius(sim, tree, mid),
                threshold,
                effective_required
            )
        )

        # fallback
        if eps is None:
            relaxed_thresh = max(1, threshold // 2)
            relaxed_prop = effective_required // 2
            eps = binary_search_condition(
                min_dist, max_dist,
                lambda mid: condition_formulation(
                    count_points_within_radius(sim, tree, mid),
                    relaxed_thresh,
                    relaxed_prop
                )
            )

        if eps is None:
            eps = max_dist

        # --------- SINGLE evaluation (UPDATED PART) ----------
        counts = count_points_within_radius(sim, tree, eps)
        return counts.astype(np.float64)


    if len(X) <= 3 * n_neighbors:
        sim = umap_graph_similarity(X)
        return compute_weights_from_similarity(sim, X)

    effective_batch_size = min(batch_size, max(n_neighbors * 3, 100))
    total_batches = (len(X) + effective_batch_size - 1) // effective_batch_size

    weights_all = []

    for batch_idx in range(total_batches):
        start = batch_idx * effective_batch_size
        end = min(len(X), start + effective_batch_size)
        X_batch = X[start:end]
        sim = umap_graph_similarity(X_batch)
        weights_batch = compute_weights_from_similarity(sim, X_batch)
        weights_all.append(weights_batch)

    return np.concatenate(weights_all)



@njit(fastmath=True, parallel=True)
def shift_data(X, indices, weights, learning_rate):

    n, k = indices.shape
    d = X.shape[1]

    revised_d = np.empty_like(X)
    change = np.empty(n)

    for i in prange(n):

        denom = 0.0
        for j in range(k):
            denom += weights[indices[i, j]]
        if denom < 1e-6:
            denom = 1e-6

        for t in range(d):
            acc = 0.0
            for j in range(k):
                acc += weights[indices[i, j]] * X[indices[i, j], t]
            revised_d[i, t] = acc / denom

        dist = 0.0
        for t in range(d):
            diff = revised_d[i, t] - X[i, t]
            dist += diff * diff
        dist = np.sqrt(dist)
        change[i] = dist

        if dist > 1e-6:
            scale = learning_rate * dist
            for t in range(d):
                revised_d[i, t] = X[i, t] + scale * (revised_d[i, t] - X[i, t]) / dist
        else:
            revised_d[i] = X[i]

    return revised_d, change


def get_shift_fast(X, k, nbd_sample_count_threshold,
                   learning_rate, max_iters_shift,
                   shift_threshold, return_weights=False):

    weights = get_empirical_weights(
        X,
        nbd_sample_count_threshold=nbd_sample_count_threshold,
        satisfiability_proportion=0.3,
        batch_size=1000
    )

    shifted_dataset = X.copy()
    total_distance = np.zeros(X.shape[0])

    for _ in range(max_iters_shift):

        index = NNDescent(
            shifted_dataset,
            n_neighbors=k,
            metric="euclidean",
            random_state=42
        )

        indices, _ = index.neighbor_graph
        indices = indices.astype(np.int64)

        revised_d, change = shift_data(
            shifted_dataset, indices, weights, learning_rate
        )

        total_distance += change
        shifted_dataset = revised_d

        if change.mean() < shift_threshold:
            break

    if return_weights:
        return shifted_dataset, weights, total_distance
    return shifted_dataset, total_distance


def mean_shift_manifold_learning(
    X, k=30, nbd_sample_count_threshold=30,
    learning_rate=.3, max_iters_shift=10,
    shift_threshold=1e-4, return_weights=False
):

    if return_weights:
        return get_shift_fast(
            X, k, nbd_sample_count_threshold,
            learning_rate, max_iters_shift,
            shift_threshold, True
        )

    return get_shift_fast(
        X, k, nbd_sample_count_threshold,
        learning_rate, max_iters_shift,
        shift_threshold
    )



class MSML:
    def __init__(
        self, seed: int,
        model_name: str = 'MSML',
        k=100,
        nbd_sample_count_threshold=70,
        learning_rate=0.1,
        max_iters_shift=6,
        shift_threshold=0.003,
        anomalyThreshold=0.22,
        scaler=StandardScaler()
    ):
        self.k = k
        self.nbd_sample_count_threshold = nbd_sample_count_threshold
        self.learning_rate = learning_rate
        self.max_iters_shift = max_iters_shift
        self.shift_threshold = shift_threshold
        self.anomalyThreshold = anomalyThreshold
        self.scaler = scaler
        self.seed = seed
        self.utils = Utils()
        self.model_name = model_name

    def fit(self, X_train, y_train=None):
        return self

    def predict_score(self, X):
        _, total_distance = mean_shift_manifold_learning(
            X,
            self.k,
            self.nbd_sample_count_threshold,
            self.learning_rate,
            self.max_iters_shift,
            self.shift_threshold
        )
        total_distance = self.scaler.fit_transform(total_distance.reshape(-1, 1))
        return expit(total_distance).squeeze()


if __name__ == "__main__":

    utils = Utils()
    utils.download_datasets()

    pipeline = RunPipeline(
        suffix='ADBench',
        parallel='unsupervise',
        realistic_synthetic_mode='local',
        noise_type=None
    )

    results = pipeline.run(clf=MSML)
    pd.DataFrame(results).to_csv('adbench/result/MSDE_single_radius_local.csv', index=False)

    # results2 = pipeline.run()
    # pd.DataFrame(results2).to_csv('adbench/result/benchmarks23.csv', index=False)


if there is any question while downloading datasets, we suggest you to download it from the website:
https://github.com/Minqi824/ADBench/tree/main/adbench/datasets
如果您在中国大陆地区，请使用链接：
https://jihulab.com/BraudoCC/ADBench_datasets/
100% [................................................................................] 3852 / 3852

100%|███████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 335.73it/s]

CIFAR10_0.npz already exists. Skipping download...
CIFAR10_1.npz already exists. Skipping download...
CIFAR10_2.npz already exists. Skipping download...
CIFAR10_3.npz already exists. Skipping download...
CIFAR10_4.npz already exists. Skipping download...
CIFAR10_5.npz already exists. Skipping download...
CIFAR10_6.npz already exists. Skipping download...
CIFAR10_7.npz already exists. Skipping download...
CIFAR10_8.npz already exists. Skipping download...
CIFAR10_9.npz already exists. Skipping download...
FashionMNIST_0.npz already exists. Skipping download...
FashionMNIST_1.npz already exists. Skipping download...
FashionMNIST_2.npz already exists. Skipping download...
FashionMNIST_3.npz already exists. Skipping download...
FashionMNIST_4.npz already exists. Skipping download...
FashionMNIST_5.npz already exists. Skipping download...
FashionMNIST_6.npz already exists. Skipping download...
FashionMNIST_7.npz already exists. Skipping download...
FashionMNIST_8.npz already exists. Skippin

subsampling for dataset 11_donors...
current noise type: None
{'Samples': 10000, 'Features': 10, 'Anomalies': np.int64(618), 'Anomalies Ratio(%)': np.float64(6.18)}
subsampling for dataset 11_donors...
current noise type: None
{'Samples': 10000, 'Features': 10, 'Anomalies': np.int64(578), 'Anomalies Ratio(%)': np.float64(5.78)}
subsampling for dataset 11_donors...
current noise type: None
{'Samples': 10000, 'Features': 10, 'Anomalies': np.int64(597), 'Anomalies Ratio(%)': np.float64(5.97)}
current noise type: None
{'Samples': 1941, 'Features': 27, 'Anomalies': np.int32(673), 'Anomalies Ratio(%)': np.float64(34.67)}
current noise type: None
{'Samples': 1941, 'Features': 27, 'Anomalies': np.int32(673), 'Anomalies Ratio(%)': np.float64(34.67)}
current noise type: None
{'Samples': 1941, 'Features': 27, 'Anomalies': np.int32(673), 'Anomalies Ratio(%)': np.float64(34.67)}
subsampling for dataset 13_fraud...
current noise type: None
{'Samples': 10000, 'Features': 29, 'Anomalies': np.int64(15)

0it [00:00, ?it/s]

...

Current experiment parameters: ('9_census', 0.0, np.int64(1)), model: Customized, metrics: {'aucroc': np.float64(1.0), 'aucpr': np.float64(1.0), 'p_at_n': np.float64(1.0), 'adj_p_at_n': np.float64(1.0), 'adj_ap': np.float64(1.0)}, fitting time: 9.5367431640625e-07, inference time: 8.517272233963013
subsampling for dataset 9_census...
current noise type: None
{'Samples': 10000, 'Features': 500, 'Anomalies': np.int64(628), 'Anomalies Ratio(%)': np.float64(6.28)}


1082it [25:16,  4.70s/it]

Model: Customized, AUC-ROC: 0.9999943252322872, AUC-PR: 0.9999164531580707
Current experiment parameters: ('9_census', 0.0, np.int64(2)), model: Customized, metrics: {'aucroc': np.float64(0.9999943252322872), 'aucpr': np.float64(0.9999164531580707), 'p_at_n': np.float64(0.9946808510638298), 'adj_p_at_n': np.float64(0.9943252322871583), 'adj_ap': np.float64(0.999910867522835)}, fitting time: 1.1920928955078125e-06, inference time: 8.554775714874268
subsampling for dataset 9_census...
current noise type: None
{'Samples': 10000, 'Features': 500, 'Anomalies': np.int64(652), 'Anomalies Ratio(%)': np.float64(6.52)}
Model: Customized, AUC-ROC: 0.9949052374159364, AUC-PR: 0.9535912737163313


1104it [26:31,  1.44s/it]

Current experiment parameters: ('9_census', 0.0, np.int64(3)), model: Customized, metrics: {'aucroc': np.float64(0.9949052374159364), 'aucpr': np.float64(0.9535912737163313), 'p_at_n': np.float64(0.8673469387755102), 'adj_p_at_n': np.float64(0.8580744708725144), 'adj_ap': np.float64(0.9503472971287426)}, fitting time: 1.1920928955078125e-06, inference time: 8.777536392211914
